# 05 · Paged KV Cache 与前缀复用

上一章留了两个坑，这一章一起填掉：

1. **填充浪费**：一个 batch 里混进一条长序列，整个 batch 都要按最长的那条分配显存。
2. **重复计算**：大量请求共享同一段 prompt 前缀（系统提示、检索模板、few-shot 示例），却各自重算一遍。

vLLM 的 PagedAttention 同时解决了这两个问题。这一章把它的核心机制亲手实现一遍。

In [ ]:
# ===== 引导单元：环境检查 + 测量工具 + MiniGPT（每章自带，直接运行）=====
# 说明：本单元在每个 notebook 里都有一份完整副本，目的是让任何一个 notebook
#       都能在 Colab 里零配置独立运行。想改模型结构，请改 tools/build_notebooks.py
#       里的 SETUP_CODE，然后重跑编译脚本。
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MiniGPT 只有 2700 万参数，用 float16 跑在 GPU 上；CPU 上 float16 很慢，用 float32
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32


def sync():
    """GPU 是异步执行的，计时前必须同步，否则测到的是下发时间不是执行时间。"""
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def bench(fn, warmup=3, iters=10):
    """返回单次调用的平均耗时（毫秒）。warmup 用来排除首次 kernel 编译等开销。"""
    for _ in range(warmup):
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    sync()
    return (time.perf_counter() - t0) / iters * 1000.0


def peak_mem_mb():
    """当前 CUDA 峰值显存占用（MB）。"""
    if DEVICE != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated() / 1024 ** 2


def reset_peak():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()


class Config:
    def __init__(self, vocab_size=50257, block_size=1024, n_layer=4, n_head=6, n_embd=384):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head


class CausalSelfAttention(nn.Module):
    """因果自注意力，支持 KV cache。

    past_kv 传入历史的 (k, v)，本步只为新 token 计算 Q/K/V，然后拼在历史后面。
    返回 (输出, 更新后的 (k, v))，其中 k/v 的 shape 是 (B, n_head, 总长度, head_dim)。
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.head_dim
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x, past_kv=None, attn_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)

        S = k.size(2)  # 总长度 = 历史 + 本步新增
        if attn_mask is None:
            # 默认因果掩码：本步第 i 个 query 的绝对位置是 S-T+i，只能看见 <= 它的 key
            mask = torch.ones(T, S, device=x.device).tril(diagonal=S - T).bool()
        else:
            # 外部传入的掩码，用于一个 batch 里混合不同进度的序列（第 04、06 章）
            mask = attn_mask
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), (k, v)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, past_kv=None, attn_mask=None):
        h, present = self.attn(self.ln_1(x), past_kv, attn_mask)
        x = x + h
        x = x + self.mlp(self.ln_2(x))
        return x, present


class MiniGPT(nn.Module):
    """极简 GPT，结构与 Llama 同源：pre-norm + 因果注意力 + 4 倍扩张 MLP + 权重共享。

    与 Llama 的两处差异：
      - 用可学习位置编码代替 RoPE（简化实现，不影响调度实验的结论）
      - 没有 GQA（本仓库是 MHA，第 03 章会手工比较两者的 KV cache 大小）
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # 权重共享，省一份 embedding 参数

        def init(m):
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

        self.apply(init)

    def forward(self, idx, past_kvs=None, pos_offset=0, attn_mask=None):
        """idx: (B, T) 的 token id。

        past_kvs: 长度等于层数的列表，每项是 (k, v)；None 表示从零开始（prefill）。
        pos_offset: 本次输入的第一个 token 的绝对位置。传 int 表示整个 batch 用同一个
                    偏移；传 shape (B,) 的张量表示每条序列各用各的偏移——当 batch 里
                    混合了不同进度的请求时必须这样传。
        attn_mask: 可选的自定义注意力掩码，用于屏蔽填充位。
        """
        B, T = idx.shape
        if torch.is_tensor(pos_offset):
            pos = pos_offset.view(B, 1) + torch.arange(T, device=idx.device)[None, :]
        else:
            pos = torch.arange(pos_offset, pos_offset + T, device=idx.device)[None, :].expand(B, T)
        x = self.wte(idx) + self.wpe(pos)

        presents = []
        for i, blk in enumerate(self.blocks):
            past = None if past_kvs is None else past_kvs[i]
            x, present = blk(x, past, attn_mask)
            presents.append(present)
        return self.lm_head(self.ln_f(x)), presents

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def build_model(seed=0, device=DEVICE, dtype=DTYPE, **kw):
    torch.manual_seed(seed)
    cfg = Config(**kw)
    model = MiniGPT(cfg).to(device=device, dtype=dtype)
    return model.eval()


@torch.no_grad()
def generate_naive(model, idx, max_new_tokens):
    """不用 KV cache：每一步都把完整序列重新算一遍（O(n^2) 重算）。"""
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.cfg.block_size:])
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens):
    """用 KV cache：prompt 只 prefill 一次，之后每步只喂 1 个 token。"""
    logits, past = model(idx)
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    out = [nxt]
    pos = idx.size(1)
    for _ in range(max_new_tokens - 1):
        logits, past = model(nxt, past_kvs=past, pos_offset=pos)
        pos += 1
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out.append(nxt)
    return torch.cat([idx] + out, dim=1)


def kv_bytes(n_layer, n_kv_head, head_dim, seq_len, batch=1, dtype_bytes=2):
    """KV cache 字节数。注意是 2（K 和 V 各一份）。"""
    return 2 * n_layer * n_kv_head * head_dim * seq_len * batch * dtype_bytes


print(f"引导单元加载完成 | device={DEVICE} dtype={DTYPE} torch={torch.__version__}")
# ===== 引导单元结束 =====

## 一、核心思路：把 KV cache 当虚拟内存管

操作系统怎么解决"进程需要连续内存但物理内存会碎片"的问题？**分页**——进程看到的是连续的虚拟地址，实际映射到任意物理页，靠页表翻译。

PagedAttention 是同一个思路：

| 操作系统 | PagedAttention |
|---|---|
| 物理页 | KV block（固定大小，比如 16 个 token） |
| 页表 | block_table（记录这条序列用了哪些 block） |
| 进程 | 一条请求序列 |
| 共享库（多个进程共享代码页） | **前缀复用**（多条请求共享同一批 block） |

关键收益：序列的 KV 不再需要物理连续，也**不需要为了对齐而填充**。按需分配，用多少给多少。

In [ ]:
class BlockManager:
    """KV cache 块分配器。这是 PagedAttention 里最关键的一个组件。"""

    def __init__(self, num_blocks, block_size):
        self.block_size = block_size
        self.num_blocks = num_blocks
        self.free_blocks = list(range(num_blocks))
        self.tables = {}      # seq_id -> [block_id, ...]
        self.ref_count = {}   # block_id -> 被几条序列引用（前缀共享靠它）

    def blocks_needed(self, n_tokens):
        return math.ceil(n_tokens / self.block_size)

    def allocate(self, seq_id, n_tokens):
        """按需分配。已有够用就不动，不够才补。"""
        need = self.blocks_needed(n_tokens)
        have = len(self.tables.get(seq_id, []))
        if need > have:
            extra = need - have
            if extra > len(self.free_blocks):
                raise MemoryError(f"剩余 block 不足：需要 {extra}，只剩 {len(self.free_blocks)}")
            got = [self.free_blocks.pop() for _ in range(extra)]
            for b in got:
                self.ref_count[b] = 1
            self.tables.setdefault(seq_id, []).extend(got)
        return self.tables[seq_id]

    def free(self, seq_id):
        """释放 = 引用计数减一，减到 0 才真正归还。"""
        for b in self.tables.pop(seq_id, []):
            self.ref_count[b] -= 1
            if self.ref_count[b] == 0:
                self.free_blocks.append(b)

    def share_prefix(self, dst_id, src_id):
        """把 src 的 block_table 直接给 dst，引用计数加一——零拷贝的前缀复用。"""
        blocks = list(self.tables[src_id])
        self.tables[dst_id] = blocks
        for b in blocks:
            self.ref_count[b] += 1
        return blocks

    @property
    def used_blocks(self):
        return self.num_blocks - len(self.free_blocks)


bm = BlockManager(num_blocks=64, block_size=16)
bm.allocate("reqA", 100)
print(f"reqA 100 个 token → 占 {len(bm.tables['reqA'])} 个 block（112 token 的容量）")
bm.share_prefix("reqB", "reqA")
print(f"reqB 共享 reqA 前缀后，总占用仍是 {bm.used_blocks} 个 block（零拷贝）")
bm.free("reqA")
print(f"reqA 释放后仍占用 {bm.used_blocks} 个（因为 reqB 还引用着）")
bm.free("reqB")
print(f"reqB 也释放后占用 {bm.used_blocks} 个")

## 二、分页省了多少：和"预留最大长度"对比

在没有分页的系统里，序列往往要**预留整个最大长度**的连续空间。分页则按需增长。差距有多大？

In [ ]:
def utilization(strategy, lens, block_size=16, max_len=4096):
    ideal = sum(lens)
    if strategy == "reserve":      # 每条序列预留最大长度
        allocated = len(lens) * max_len
    elif strategy == "paged":      # 按需分配，向上取整到 block
        allocated = sum(math.ceil(L / block_size) * block_size for L in lens)
    else:
        allocated = ideal
    return ideal / allocated


workloads = {
    "8 条短请求 (100 token)": [100] * 8,
    "8 条长请求 (4K 上下文)": [4096] * 8,
    "长短混合 (100~4K)": [100, 200, 400, 800, 1600, 3200, 4096, 4096],
    "8 条刚起步 (10 token)": [10] * 8,
}

print(f"{'场景':<26}{'预留最大长度':>14}{'分页按需':>12}")
print("-" * 54)
for name, lens in workloads.items():
    print(f"{name:<26}{utilization('reserve', lens) * 100:>13.1f}%{utilization('paged', lens) * 100:>11.1f}%")

print()
print("注意最后一行：请求刚起步、只用了 10 个 token 时，预留式方案浪费了 99.8% 的空间。")
print("分页方案只浪费 block 内部的取整部分。线上大量请求同时处在不同阶段，")
print("这就是 PagedAttention 能把并发做上去的原因。")

## 三、block size 怎么选

分页不是免费的：block 末尾用不满的部分就是**内部碎片**。block 越小碎片越少，但 block 表越长、寻址开销越大。

In [ ]:
SEQ_LEN = 8192
print(f"以一条 {SEQ_LEN} token 的序列为例：\n")
print(f"{'block_size':>12}{'平均内部碎片':>18}{'block 表项数':>16}")
print("-" * 48)
for bs in [1, 4, 8, 16, 32, 64, 128, 256]:
    avg_frag = (bs - 1) / 2          # 长度均匀分布时的平均浪费
    print(f"{bs:>12}{avg_frag:>14.1f} token{SEQ_LEN // bs:>15}")

print()
print("怎么权衡：")
print("  block_size 太小 → block 表很长，每次 attention 要遍历更多块，索引开销上升")
print("  block_size 太大 → 内部碎片严重，短请求的显存被浪费")
print("  16 是主流默认值（vLLM 默认就是 16）：平均碎片 7.5 个 token，8K 序列的表 512 项，两边都还能接受")
print()
print("面试时能说出'这是碎片和寻址开销的折中，而且 kernel 实现对这个值有约束'，就高出一个层次了。")

## 四、前缀复用：命中判定比你想象的严格

前缀缓存不是"内容相似就命中"，而是**按 block 做链式哈希，必须从第一个 block 起连续匹配**：

```
hash(block 0) = H(-1,        tokens[0:B])
hash(block 1) = H(hash(0),   tokens[B:2B])
hash(block i) = H(hash(i-1), tokens[iB:(i+1)B])
```

为什么要链式？如果只哈希 block 内容，那么相同内容出现在**不同位置**时会被误判为可复用——但它前面的上下文不同，KV 完全不同，复用就会算错。

In [ ]:
def block_hashes(token_ids, block_size):
    """链式哈希。末尾不足一个 block 的 token 不参与——因为它还会继续增长。"""
    tokens = list(token_ids)
    parent, out = -1, []
    for i in range(0, len(tokens) - block_size + 1, block_size):
        h = hash((parent, tuple(tokens[i:i + block_size])))
        out.append(h)
        parent = h
    return out


def matched_blocks(a, b, block_size):
    """返回两条序列从开头起连续匹配的 block 数。"""
    ha, hb = block_hashes(a, block_size), block_hashes(b, block_size)
    n = 0
    for x, y in zip(ha, hb):
        if x != y:
            break
        n += 1
    return n


BS = 16
BASE = list(range(1000, 1128))    # 128 token 的公共部分 = 8 个 block

base = BASE + [1, 2, 3, 4]
variants = {
    "同前缀，后缀不同": BASE + [9, 8, 7, 6],
    "前缀后追加变量": BASE + [555] + [1, 2, 3],
    "变量插在最开头": [777] + BASE,
    "变量插在第 64 token 后": BASE[:64] + [888] + BASE[64:],
}

print(f"block_size = {BS}，第一条序列共 {len(block_hashes(base, BS))} 个可缓存 block\n")
print(f"{'变体':<26}{'命中block':>10}{'判定':>16}")
print("-" * 52)
for name, seq in variants.items():
    m = matched_blocks(base, seq, BS)
    verdict = "高" if m >= 6 else ("低" if m > 0 else "完全失效")
    print(f"{name:<26}{m:>10}{verdict:>16}")

**这张表就是 prefix cache 的全部行为规律：**

- 后缀不同不影响命中——这很好，检索到的文档本来就不一样。
- 前缀后追加内容不影响命中——前面的 block 已经完整且固定了。
- **变量插在最开头，命中率直接归零**。哪怕后面 100 个 token 完全相同，第一个 block 变了，链式哈希全断。
- 变量插在中间，只有它之前的 block 能命中。

所以有一条工程铁律：**system prompt、指令模板、few-shot 放最前面，变量放最后面。**

上线前值得做一次审计：把线上 prompt 的各个字段按位置排一排，看哪个字段会让缓存整段失效。这通常是**改一行位置换来 30% 成本下降**的优化。

## 五、验证前缀复用算得对

复用前缀的 KV 和整体重算，结果必须一致。这个验证不能省——复用错了不会报错，只会悄悄让输出变差。

In [ ]:
model = build_model(block_size=4096)
P, S = 512, 128

prefix_ids = torch.randint(0, model.cfg.vocab_size, (1, P), device=DEVICE)
suffix_ids = torch.randint(0, model.cfg.vocab_size, (1, S), device=DEVICE)

logits_full, _ = model(torch.cat([prefix_ids, suffix_ids], dim=1))       # 整体重算
_, prefix_past = model(prefix_ids)
logits_reuse, _ = model(suffix_ids, past_kvs=prefix_past, pos_offset=P)  # 复用前缀 KV

diff = (logits_full[:, -1] - logits_reuse[:, -1]).abs().max().item()
same_token = torch.equal(logits_full[:, -1].argmax(-1), logits_reuse[:, -1].argmax(-1))

print(f"最后一位 logits 最大差异: {diff:.2e}")
print(f"argmax 结果一致        : {same_token}")
print()
print("差异应该在 1e-3 量级以下（浮点运算顺序不同导致的舍入），但 argmax 必须完全一致。")
print("如果 argmax 不一致，说明位置偏移或掩码算错了——这是复用前缀时最容易出的 bug。")

## 六、复用能省多少计算

In [ ]:
def prefill_cost(n_reqs, prefix_len, suffix_len, reuse=True):
    """需要计算的 token 数（prefill 计算量正比于 token 数）。"""
    if reuse:
        return prefix_len + n_reqs * suffix_len
    return n_reqs * (prefix_len + suffix_len)


N, P, Sfx = 8, 512, 64
no_reuse = prefill_cost(N, P, Sfx, reuse=False)
with_reuse = prefill_cost(N, P, Sfx, reuse=True)

print(f"{N} 条请求，每条 {P} token 共享前缀 + {Sfx} token 独立后缀\n")
print(f"不复用：{no_reuse:>6} token")
print(f"复用  ：{with_reuse:>6} token")
print(f"省下  ：{(1 - with_reuse / no_reuse) * 100:>5.1f}% 的 prefill 计算量")
print()
print("前缀越长、请求越密，收益越大。当共享前缀占到 prompt 的 90%（比如 RAG 里塞了整段检索模板），")
print("这就是数量级的差距。")

## 七、面试话术

**问：PagedAttention 解决什么问题？** 分三个层次答：

1. **消除外部碎片**：KV 不再要求物理连续，按 block 按需分配。
2. **消除对齐填充**：不同长度的序列能同 batch 跑，不需要补齐——上一章实测过，一条长序列混进来会让整个 batch 按最长分配。
3. **支持零拷贝前缀共享**：block 按引用计数共享，多条请求复用同一份前缀 KV。

**问：block size 怎么选？** 内部碎片和寻址开销的折中。小了碎片少但表长、开销大；大了反过来。主流 16。

**问：prefix cache 什么情况会完全失效？** **前缀第一个 block 就不同**。最常见原因是 prompt 把变量（用户 query、时间戳、请求 ID）放在了最前面。链式哈希必须从头连续匹配，第一个 block 断了后面全断。

**作业**

1. 把 `BS` 改成 32 和 8，重跑哈希命中实验，观察"变量插在中间"那个用例的命中 block 数怎么变。
2. 回忆一个你线上真实遇到的场景：有没有哪段 prompt 因为字段顺序导致缓存失效？
3. 思考题：前缀共享用引用计数，那如果一条序列要**修改**共享 block 里的内容怎么办？（提示：写时复制 COW）

**下一章**：长 prefill 会阻塞其他请求，怎么把它切碎混进 decode 里跑。